In [1]:
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.llms import HuggingFaceHub
from langchain.chains import ConversationalRetrievalChain

In [2]:
def extract_and_vectordb(pdf_file):
    reader = PdfReader(pdf_file)
    raw_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            raw_text += text
    
    text_splitter = RecursiveCharacterTextSplitter()
    texts = text_splitter.split_text(raw_text)
    
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vec_db = FAISS.from_texts(texts, embeddings)
    
    return vec_db

In [3]:
template = """
You are a helpful assistant who provides accurate and straightforward answers based on the given document.\n
Given the following document:\n{context}\n\nAnswer the question:\n{question}\n\n
Answer:
"""
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

In [4]:
from langchain.memory import ConversationBufferMemory

In [5]:
def initialize_chain(vector_db, model_name, api_key, temperature):
    llm = HuggingFaceHub(
        repo_id=model_name,
        model_kwargs={"temperature": temperature, "max_tokens": 512},
        huggingfacehub_api_token=api_key,
    )
    retriever = vector_db.as_retriever(search_kwargs={"k": 5})
    #memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    conversation_chain = ConversationalRetrievalChain.from_llm(
        llm, 
        retriever,
        combine_docs_chain_kwargs={"prompt": prompt},
        #memory=memory
        )
    return conversation_chain

In [6]:
def postprocess(response):
    query_index = response.rfind("Answer:")
    if query_index != -1:
        answer = response[query_index + len("Answer:"):].strip()
        answer = answer.lstrip("\n\"")
        return answer
    else:
        return ""

In [7]:
def get_the_answer(conversation_chain, question, chat_history):
    conv_response = conversation_chain.invoke({"question": question, "chat_history": chat_history})
    response = postprocess(conv_response['answer'])
    return response

In [8]:
vector_db = extract_and_vectordb("ai_doc.pdf")

HUGGINGFACEHUB_API_TOKEN = ""
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

conversation_chain = initialize_chain(
    vector_db=vector_db,
    model_name=model_name,
    api_key=HUGGINGFACEHUB_API_TOKEN,
    temperature=0.7
)

c:\Users\ahmed_3hijq3m\AppData\Local\Programs\Python\Python312\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


C:\Users\ahmed_3hijq3m\AppData\Local\Temp\ipykernel_14560\1478875865.py:2: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(


In [9]:
chat_history = []

In [10]:
question = "Hi, How are you?"

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

I am an AI-powered assistant designed to help you with your questions. How may I assist you today? Let me know if you have any questions about the document you provided, specifically related to the role of artificial intelligence in data analytics.


In [11]:
question = "I want to ask you some questions about the document, can you help me?"

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

Of course! I'd be happy to help answer your questions about the document. Let me know what specific questions you have, and I'll do my best to provide accurate and informative answers.


In [12]:
question = "what is the document main topic?"

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

The main topic of the document is the impact of Artificial Intelligence (AI) on data analytics, particularly its role in enhancing efficiency, accuracy, and predictive power in various industries.


In [13]:
question = "what are the document sections?"

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

The document sections are:
1. Title: The Impact of Artificial Intelligence on Data Analytics
2. Abstract
3. Introduction
4. Body (with subsections):
   a) Machine Learning in Data Analytics
   b) Deep Learning and Neural Networks
   c) AI-Driven Predictive Analytics
   d) Automation of Data Processing
5. Conclusion
6. Summary


In [14]:
question = "what are the body sub-sections?"

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

The body sub-sections of the paper are:
1. Machine Learning in Data Analytics
2. Deep Learning and Neural Networks
3. AI-Driven Predictive Analytics
4. Automation of Data Processing

Each sub-section discusses a specific aspect of how Artificial Intelligence (AI) is used in data analytics.


In [15]:
question = "tell me more about the first body sub-section."

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

In the first body sub-section, the document discusses the role of Machine Learning (ML) in data analytics. It explains that machine learning is a vital part of AI and plays a significant role in automating data analysis by creating models that predict trends, classify data, and make decisions. The sub-section further elaborates that machine learning allows for the identification of hidden patterns in complex datasets that would be difficult for human analysts to spot, thereby enhancing the efficiency and accuracy of data


In [16]:
question = "what is the first sentense of the document Introduction?"

response = get_the_answer(
    conversation_chain=conversation_chain,
    question=question,
    chat_history=chat_history
    )

print(response)

Artificial Intelligence (AI) has emerged as one of the most transformative forces in the digital era, impacting various sectors, including healthcare, finance, marketing, and more.


In [17]:
chat_history

[]